# 02 — Scaled Dot-Product Attention
### Solution Notebook

---

In [ ]:
import sys, os, math
sys.path.insert(0, os.path.abspath('../..'))

import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional, Tuple

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Part A — Implement Scaled Dot-Product Attention

In [ ]:
def my_attention(
    query: torch.Tensor,
    key:   torch.Tensor,
    value: torch.Tensor,
    mask:  Optional[torch.Tensor] = None,
) -> Tuple[torch.Tensor, torch.Tensor]:
    d_k = query.size(-1)

    # Step 1 — raw scores
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)

    # Step 2 — mask
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    # Step 3 — softmax
    attn_weights = F.softmax(scores, dim=-1)

    # Step 4 — weighted values
    output = torch.matmul(attn_weights, value)

    return output, attn_weights


# Correctness test
B, H, S, dk = 2, 4, 6, 16
q = torch.randn(B, H, S, dk)
k = torch.randn(B, H, S, dk)
v = torch.randn(B, H, S, dk)
out, w = my_attention(q, k, v)
print(f'Output shape  : {out.shape}')
print(f'Weights shape : {w.shape}')
print(f'Weights sum   : {w.sum(-1).mean():.4f}  ← should be 1.0')

## Part A2 — Scaling Effect on Softmax Entropy

In [ ]:
def softmax_entropy(logits):
    probs = F.softmax(logits, dim=-1)
    log_p = torch.log2(probs + 1e-9)
    return -(probs * log_p).sum(-1).mean().item()

dims = [8, 16, 32, 64, 128, 256, 512]
entropies_unscaled, entropies_scaled = [], []
for d in dims:
    q = torch.randn(64, d)
    k = torch.randn(64, d)
    scores = q @ k.T
    entropies_unscaled.append(softmax_entropy(scores))
    entropies_scaled.append(softmax_entropy(scores / math.sqrt(d)))

plt.figure(figsize=(8, 4))
plt.plot(dims, entropies_unscaled, 'o-', color='tomato',   label='No scaling')
plt.plot(dims, entropies_scaled,   's-', color='steelblue', label='Scaled (÷√d_k)')
plt.xlabel('d_k (head dimension)')
plt.ylabel('Softmax entropy (bits)')
plt.title('Effect of Scaling on Attention Distribution')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\nObservation: without scaling, entropy → 0 as d_k grows.')
print('This means the model is effectively forced to hard-attend to a single token,')
print('which kills gradient flow.')

## Part B — Causal Masking

In [ ]:
def make_causal_mask(seq_len, device=None):
    # Lower-triangular: position i can attend to positions 0..i
    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    return mask.view(1, 1, seq_len, seq_len)

mask = make_causal_mask(6)
plt.figure(figsize=(4, 3))
plt.imshow(mask.squeeze(), cmap='Blues', vmin=0, vmax=1)
plt.title('Causal Mask'); plt.xlabel('Key pos'); plt.ylabel('Query pos')
plt.colorbar(); plt.tight_layout(); plt.show()

In [ ]:
# B2 — verify causal attention
seq = 8
q = k = v = torch.randn(1, 1, seq, 32)
mask = make_causal_mask(seq)
out, w = my_attention(q, k, v, mask=mask)

w_np = w[0, 0].detach().numpy()
upper_sum = np.triu(w_np, k=1).sum()
print(f'Upper triangle sum (should be ~0): {upper_sum:.6f}')

plt.figure(figsize=(5, 4))
plt.imshow(w_np, cmap='Blues', vmin=0, vmax=1)
plt.title('Causal Attention Weights')
plt.xlabel('Key pos'); plt.ylabel('Query pos')
plt.colorbar(); plt.tight_layout(); plt.show()

## Part C — Compare with Library

In [ ]:
from src.models.attention import scaled_dot_product_attention

B, H, S, dk = 2, 4, 10, 32
q = torch.randn(B, H, S, dk)
k = torch.randn(B, H, S, dk)
v = torch.randn(B, H, S, dk)

lib_out, lib_w = scaled_dot_product_attention(q, k, v)
my_out, my_w   = my_attention(q, k, v)

print(f'Outputs match : {torch.allclose(lib_out, my_out, atol=1e-5)}')
print(f'Weights match : {torch.allclose(lib_w,   my_w,   atol=1e-5)}')

## Part D — Text Attention Visualisation

In [ ]:
sentence = "the cat sat on the mat"
chars = list(sentence)
vocab = {c: i for i, c in enumerate(sorted(set(chars)))}
tokens = torch.tensor([vocab[c] for c in chars])

embed_dim = 16
embedding = nn.Embedding(len(vocab), embed_dim)
x = embedding(tokens).unsqueeze(0)

Wq = nn.Linear(embed_dim, embed_dim, bias=False)
Wk = nn.Linear(embed_dim, embed_dim, bias=False)
Wv = nn.Linear(embed_dim, embed_dim, bias=False)

q = Wq(x).unsqueeze(1)
k = Wk(x).unsqueeze(1)
v = Wv(x).unsqueeze(1)

_, w = my_attention(q, k, v)
w_np = w[0, 0].detach().numpy()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(w_np, cmap='Blues', vmin=0)
ax.set_xticks(range(len(chars))); ax.set_xticklabels(chars, fontfamily='monospace')
ax.set_yticks(range(len(chars))); ax.set_yticklabels(chars, fontfamily='monospace')
ax.set_xlabel('Key (attended to)'); ax.set_ylabel('Query (attending)')
ax.set_title(f'Attention weights — "{sentence}"')
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

## Summary

The four-step recipe:

```python
scores = (Q @ K.T) / sqrt(d_k)   # 1. dot-product + scale
scores = mask_fill(scores, -inf)  # 2. optional mask
weights = softmax(scores, dim=-1) # 3. normalise
output  = weights @ V             # 4. aggregate values
```

**Next:** `03_multihead_attention_starter.ipynb`